## Stage 1 Build A Minimal Agent Loop

- [x] Use an LLM API to complete a basic conversation.
- [x] Get the model to output structured JSON.
- [x] Define a tool function, e.g. `search`, `calculator`, `read_file`.
- [x] Parse the model's tool call / function call.
- [x] Execute the tool and feed the result back to the model.
- [x] Add max steps, timeout, and error handling to the agent loop.

Docs:
- [Function calling with the Gemini API](https://ai.google.dev/gemini-api/docs/function-calling?example=meeting)
- [Google Gen AI SDK](https://googleapis.github.io/python-genai/#function-calling)

In [ ]:
# 0. Setup
from google import genai

client = genai.Client(
    vertexai=True,
    project="xxxx", # use yourself one
    location="global",
)

In [2]:
# 1. LLM API call

model = "gemini-2.5-flash"
prompts = r"""
What's the time in Germany??
"""

response = client.models.generate_content(
    model=model,
    contents=prompts,
)
print(response.text)

It's currently **12:20 PM** on Friday, May 17, 2024, in Germany.

Germany observes Central European Summer Time (CEST), which is UTC+2.


In [3]:
# 2. Structured outputs

from pydantic import BaseModel
from google.genai import types
from datetime import datetime

class TimeInfo(BaseModel):
    time: datetime

response = client.models.generate_content(
    model=model,
    contents=prompts,
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=TimeInfo,
    ),
)

info = TimeInfo.model_validate_json(response.text)
print(info)

time=datetime.datetime(2024, 5, 24, 12, 30, tzinfo=TzInfo(7200))


In [4]:
# 3. Define a tool

get_time_declaration = types.FunctionDeclaration(
    name='get_time',
    description='Give the current time given the time zone.',
    parameters_json_schema={
        'type': 'object',
        'properties': {
            'timezone': {
                'type': 'string',
                'description': 'time zone in pytz format, e.g. America/Los_Angeles',
            }
        },
        'required': ['timezone'],
    },
)


def get_time(timezone: str)-> datetime:
    """Get the current date and time of given the time zone.

    Args:
    timezone

    Returns: datetime
    """
    import pytz

    timezone = pytz.timezone(timezone)
    now = datetime.now(tz=timezone)
    return now.strftime("%A, %d %B %Y, %H:%M")

In [5]:
# 4. Parse model's tool call

# Configure the client and tools
get_time_tool = types.Tool(function_declarations=[get_time_declaration])
config = types.GenerateContentConfig(tools=[get_time_tool])

# Define user prompt
contents = [
    types.Content(
        role="user", parts=[types.Part(text=prompts)]
    )
]

# Send request with function declarations
response = client.models.generate_content(
    model=model,
    contents=contents,
    config=config,
)

print(response.candidates[0].content.parts[0].function_call)

id=None args={'timezone': 'Europe/Berlin'} name='get_time' partial_args=None will_continue=None


In [6]:
# 5. Excute the tool and feed back to model

# Extract tool call details
tool_call = response.candidates[0].content.parts[0].function_call


if tool_call.name == "get_time":
    result = get_time(**tool_call.args)
    print(f"Function execution result: {result}")

# Create a function response part
function_response_part = types.Part.from_function_response(
    name=tool_call.name,
    response={"result": result},
    # id=tool_call.id, used for Gemini 3.X
)

# Append function call and result of the function execution to contents
contents.append(response.candidates[0].content) # Append the content from the model's response.
contents.append(types.Content(role="user", parts=[function_response_part])) # Append the function response

final_response = client.models.generate_content(
                    model=model,
                    config=config,
                    contents=contents,
                    )

print(f"Clock Agent output:{final_response.text}.")

Function execution result: Thursday, 28 May 2026, 18:12
Clock Agent output:The current time in Germany is 18:12 on Thursday, 28 May 2026..


In [7]:
# 6. Add max steps, timeout, and error handling to the agent loop

# max step is to limit how many times we allow the model to use tools, to prevent model's endless tool calling and expensive bills

import time

TOOLS = {"get_time": get_time}
prompts = r"""What's the time in China?"""
contents = [
    types.Content(
        role="user", parts=[types.Part(text=prompts)]
    )
]

step = 0
max_steps = 5
timeout = 10  # seconds
start_time = time.time()


while step < max_steps:
    # handle timeout
    if time.time() - start_time > timeout:
        print("Timeout reached, Stop!")
        break
        
    # Parse and excute the tool call
    response = client.models.generate_content(
        model=model,
        contents=contents,
        config=config,
    )
    tool_call_part = response.candidates[0].content.parts[0]
    contents.append(response.candidates[0].content)
    
    if tool_call_part.function_call:
        tc = tool_call_part.function_call
        print(f"[DEBUG] calling {tc.name} with args {tc.args}")   # model return based on tool declarations
        try:
            result = TOOLS[tc.name](**tc.args)
            print(f"[DEBUG] tool result: {result}")               # what does the tool return
        except Exception as e:
            print(f"[DEBUG] tool FAILED: {type(e).__name__}: {e}") # failed
            result = f"Error executing {tc.name}: {e}"

        
        contents.append(types.Content(
                role="user",
                parts=[types.Part.from_function_response(name=tc.name, response={"result": result})],
            ))
        step += 1 
        continue
    else:
        print(f"Clock Agent output: {response.text}")
        break

if step >= max_steps:
    print("MAX steps reached.")

Clock Agent output: Could you please provide a more specific city in China?


In [8]:
client.close()